# 02 - Normalization experiments

This notebook evaluates normalization strategies using the production `preprocessing.py` module. Experimental alternatives are kept inside the notebook; only measured, safe decisions should be promoted to production. Raw files are never modified.

In [ ]:
from pathlib import Path
import re
import sys
import unicodedata
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists(): PROJECT_ROOT = PROJECT_ROOT.parent
SRC = PROJECT_ROOT / 'src'
if not SRC.exists(): SRC = PROJECT_ROOT / 'code' / 'business_entity_resolution' / 'src'
sys.path.insert(0, str(SRC.resolve()))
from config import default_config
from data_loader import load_source
from preprocessing import build_normalized_columns, normalize_business_address, normalize_business_name, normalize_country, extract_tokens, extract_postal_tokens, extract_locality_tokens

CONFIG = default_config().resolved(PROJECT_ROOT)
REPORT_DIR = PROJECT_ROOT / 'artifacts' / 'reports' / 'normalization'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_ROWS = 100_000
print('Using project root:', PROJECT_ROOT)

In [ ]:
# Load configured source files through the production loader.
frames = {}
for source, filename in CONFIG.dataset.train_source_files.items():
    path = CONFIG.dataset.train_path(source)
    if path.exists():
        frame = load_source(path, CONFIG.schema)
        frames[source] = frame.head(SAMPLE_ROWS).copy()

if not frames:
    raise FileNotFoundError('No configured training source files were found')
print({source: frame.shape for source, frame in frames.items()})

In [ ]:
def simple_strategy(value, *, remove_punctuation=True, sort_words=False, remove_suffixes=False):
    text = '' if pd.isna(value) else str(value)
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode().casefold()
    text = text.replace('&', ' and ')
    if remove_punctuation: text = re.sub(r'[^\w\s]', ' ', text)
    tokens = re.sub(r'\s+', ' ', text).strip().split()
    if remove_suffixes:
        suffixes = set(CONFIG.normalization.legal_name_suffixes)
        tokens = [token for token in tokens if token not in suffixes]
    if sort_words: tokens = sorted(tokens)
    return ' '.join(tokens)

def strategy_table(frame, column, kind):
    raw = frame[column].fillna('').astype(str)
    strategies = {
        'raw': raw,
        'case_unicode': raw.map(lambda x: simple_strategy(x, remove_punctuation=False)),
        'punctuation_whitespace': raw.map(lambda x: simple_strategy(x)),
        'suffix_removed': raw.map(lambda x: simple_strategy(x, remove_suffixes=True)),
        'word_order_insensitive': raw.map(lambda x: simple_strategy(x, remove_suffixes=(kind == 'name'), sort_words=True)),
    }
    return strategies

In [ ]:
# Compare strategy-level unique counts and within-table collision rates.
comparison_rows = []
collision_rows = []
example_rows = []
for source, frame in frames.items():
    for column, kind in [(CONFIG.schema.name_column, 'name'), (CONFIG.schema.address_column, 'address'), (CONFIG.schema.country_column, 'country')]:
        strategies = strategy_table(frame, column, kind)
        raw = strategies['raw']
        for strategy, normalized in strategies.items():
            groups = pd.DataFrame({'raw': raw, 'normalized': normalized}).groupby('normalized')['raw'].nunique()
            dangerous = groups[groups > 1]
            comparison_rows.append({'source': source, 'column': column, 'strategy': strategy, 'rows': len(frame), 'raw_unique': int(raw.nunique(dropna=False)), 'normalized_unique': int(normalized.nunique(dropna=False)), 'unique_change': int(normalized.nunique(dropna=False) - raw.nunique(dropna=False)), 'collision_groups': int(len(dangerous)), 'collision_rate': float(len(dangerous) / max(1, normalized.nunique(dropna=False)))})
            if len(dangerous):
                for normalized_value in dangerous.index[:20]:
                    values = pd.DataFrame({'raw': raw, 'normalized': normalized}).query('normalized == @normalized_value')['raw'].drop_duplicates().tolist()
                    collision_rows.append({'source': source, 'column': column, 'strategy': strategy, 'normalized_value': normalized_value, 'raw_values': ' | '.join(values[:10]), 'raw_value_count': len(values)})
            changed = pd.DataFrame({'raw': raw, 'normalized': normalized})
            for _, item in changed[(changed.raw != '') & (changed.raw != changed.normalized)].drop_duplicates().head(20).iterrows():
                example_rows.append({'source': source, 'column': column, 'strategy': strategy, 'raw_value': item.raw, 'normalized_value': item.normalized})

comparison = pd.DataFrame(comparison_rows)
collisions = pd.DataFrame(collision_rows)
examples = pd.DataFrame(example_rows)
display(comparison.sort_values(['column', 'strategy', 'source']))
display(collisions.head(50))

In [ ]:
# Apply the production normalization and inspect helper columns.
production = {source: build_normalized_columns(frame, CONFIG.schema, CONFIG.normalization) for source, frame in frames.items()}
helper_columns = ['name_normalized', 'address_normalized', 'country_normalized', 'name_tokens', 'name_tokens_sorted', 'address_tokens', 'address_tokens_sorted', 'address_numeric_tokens', 'address_postal_tokens', 'address_locality_tokens']
display(production[next(iter(production))][[CONFIG.schema.name_column, CONFIG.schema.address_column, CONFIG.schema.country_column] + helper_columns].head(20))

# Measure exact normalized-name overlap across configured sources, without assuming names.
source_names = list(production)
overlap_rows = []
if len(source_names) >= 2:
    reference_names = set(production[source_names[0]]['name_normalized'].dropna())
    for source in source_names[1:]:
        values = set(production[source]['name_normalized'].dropna())
        overlap_rows.append({'reference_source': source_names[0], 'candidate_source': source, 'raw_exact_name_overlap': int(set(frames[source_names[0]][CONFIG.schema.name_column].astype(str)) & set(frames[source][CONFIG.schema.name_column].astype(str))), 'normalized_name_overlap': int(len(reference_names & values)), 'overlap_increase': int(len(reference_names & values))})
overlap = pd.DataFrame(overlap_rows)
display(overlap)

In [ ]:
# Address helper evidence: numeric, postal-like, and conservative locality extraction.
address_examples = []
for source, frame in production.items():
    address_examples.append(frame[['address_normalized', 'address_numeric_tokens', 'address_postal_tokens', 'address_locality_tokens']].assign(source=source).head(100))
address_examples = pd.concat(address_examples, ignore_index=True)
display(address_examples.head(30))

# Country values remain open-set strings; no country allow-list is applied.
country_values = pd.concat([frame['country_normalized'] for frame in production.values()]).value_counts(dropna=False).rename_axis('country_normalized').reset_index(name='count')
display(country_values)

In [ ]:
comparison.to_csv(REPORT_DIR / 'strategy_comparison.csv', index=False)
examples.to_csv(REPORT_DIR / 'examples.csv', index=False)
collisions.to_csv(REPORT_DIR / 'collision_report.csv', index=False)
overlap.to_csv(REPORT_DIR / 'overlap_summary.csv', index=False)

summary = comparison.groupby('strategy', as_index=False).agg({'unique_change':'mean', 'collision_rate':'mean', 'collision_groups':'sum'}).sort_values(['collision_rate', 'unique_change'])
summary.to_csv(REPORT_DIR / 'normalization_summary.csv', index=False)

observations = ['All transformation examples and collision groups in this report were measured from the loaded sample.', 'Original values remain in the source columns; production helpers are additive.', f'{len(collisions)} collision examples were recorded across the tested strategies.', 'A strategy should be promoted only after reviewing collision_report.csv, not solely because normalized overlap increased.', 'Country normalization is formatting-only and does not restrict values to a configured country list.', 'Postal and locality helpers are structural signals; they are not geocoding or external enrichment.']
(REPORT_DIR / 'normalization_summary.md').write_text('# Normalization Experiment Summary\n\n' + '\n'.join(f'- {item}' for item in observations) + '\n\n## Strategy aggregates\n\n' + summary.to_markdown(index=False), encoding='utf-8')
print('Saved reports to', REPORT_DIR)

## Recommendation protocol

Use the existing production configuration only where measured results show useful overlap with acceptable collision risk. Preserve `name_normalized`, `address_normalized`, and `country_normalized` alongside raw values. Keep token-sorted forms as helper features rather than replacing ordered text. Review every high-frequency collision in `collision_report.csv` before enabling more aggressive suffix, abbreviation, or word-order rules.